# MaSIF-site: predict protein interaction sites

Use a prepared protein surface to identify potential protein interaction sites.

This notebook previews the configuration before running your data. Running every cell with the default settings does not start a prediction.


## 1. Before you start

Install MaSIF using the [README installation instructions](../README.md#installation). Use a notebook kernel from your `masif_torch` environment, and obtain the checkpoint listed in the [weights guide](../weights/README.md).

Follow [Open a tutorial](README.md#open-a-tutorial) to install Jupyter and register the `masif_torch` kernel. Confirm the selected notebook kernel uses the same Python as your MaSIF installation.


In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

# Locate the checkout whether this notebook starts in tutorials/ or the project root.
start = Path.cwd().resolve()
project_root = next(
    (path for path in (start, *start.parents)
     if (path / "pyproject.toml").is_file() and (path / "examples").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open this notebook from the MaSIF_torch checkout.")
os.chdir(project_root)
print("Project:", project_root)
print("Notebook Python:", sys.executable)


## 2. Prepare your input

Provide a prepared surface NPZ file. If you are starting with a PDB structure, follow the [preparation tutorial](prepare_protein.ipynb) first. Keep the **complete protein surface** for this task.

If you already have original MaSIF precomputation arrays, pack one protein surface from the project root:

```bash
masif pack-surface --input data/precomputation/1AKJ_AB_DE --pid p1 \
  --trusted-indices --output data/site/example.npz
```

Use `p2` for the second partner. Only enable `--trusted-indices` for files you trust. The original folder must include the surface features, patch coordinates, masks and neighborhood indices.


## 3. Choose paths and settings

Replace `input_path` with your prepared packet. Choose an output path for this run. Keep `device = "cpu"` for a CPU run, or select `"cuda"` after installing a compatible GPU runtime.

Leave `RUN_TASK = False` while checking the configuration. Set it to `True` only when the input and checkpoint are ready.


In [ ]:
checkpoint = Path("weights/site/interface_prediction_backup.pt")
input_path = Path("data/site/example.npz")
output_path = Path("results/site/example.npy")
device = "cpu"
batch_size = 100
RUN_TASK = False

command = [
    sys.executable, "-m", "masif", "site",
    "--config", "examples/site.yaml",
    f"checkpoint={checkpoint}",
    f"input={input_path}",
    f"output={output_path}",
    f"device={device}",
    f"batch_size={batch_size}",
]


## 4. Preview the configuration

This step checks the command and displays your settings without reading the input or loading a checkpoint. Review the paths and device before continuing.


In [ ]:
print(shlex.join(command))
preview = subprocess.run(command + ["--cfg", "job"], check=True, capture_output=True, text=True)
print(preview.stdout)


## 5. Run your prepared input

After updating the settings, set `RUN_TASK = True` and rerun the settings, preview and run cells. The run cell checks that the input and checkpoint exist before starting.


In [ ]:
if RUN_TASK:
    for required_path in (input_path, checkpoint):
        if not required_path.is_file():
            raise FileNotFoundError(f"Required file not found: {required_path}")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(command, check=True)
    print("Saved:", output_path)
else:
    print("Preview only. Set RUN_TASK = True after preparing your files.")


## 6. Next steps

For an original precomputation folder, you can also run site prediction directly:

```bash
masif surface --checkpoint weights/site/interface_prediction_backup.pt \
  --input data/precomputation/1AKJ_AB_DE --pid p1 --trusted-indices \
  --output results/site/example.npy --device cpu
```

Continue with [MaSIF-search](search.ipynb) to compute surface fingerprints, or [MaSIF-seed](seed.ipynb) to search for candidate binders.

See the [usage manual](../USAGE.md) for other commands and configuration examples.
